# Campaña Avícola — Modelo Predictivo con PyCaret (Notebook 2/2)

Modelo de **regresión** para predecir el rendimiento de la campaña (`% total primera`) a partir de
variables de producción avícola (pesos por día de crecida, índices ICA/IEP, mortalidad). Se usa
**PyCaret** para comparar modelos automáticamente y afinar el mejor (Random Forest).

**Pipeline:** carga → definición de target/features → setup PyCaret → comparación de modelos →
Random Forest + tuning (MAPE/MAE) → evaluación.

> Requiere el dataset de producción con la variable objetivo `% total primera`, en `data/`.

## 1. Instalación e importación

In [ ]:
# %pip install pycaret   # descomentar si PyCaret no esta instalado
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pycaret.regression import (
    setup, compare_models, create_model, tune_model,
    plot_model, predict_model, evaluate_model, get_config,
)

## 2. Carga y definición del problema

In [ ]:
df_raw = pd.read_excel("data/Data para prediccion.xlsx")
print(f"Registros: {df_raw.shape[0]} | Variables: {df_raw.shape[1]}")
df_raw.head()

In [ ]:
TARGET = "% total primera"
FEATURES = ["Sexo", "Carga granja", "Día 7", "Día 14", "Día 21", "Día 28",
            "Día 35", "Día 38", "Top ICA", "Top IEP", "Top Mortalidad"]

df = df_raw[FEATURES + [TARGET]].copy()
df["Sexo"] = df["Sexo"].astype("category")
df.info()

In [ ]:
# Correlacion entre variables numericas
plt.figure(figsize=(10, 8))
sns.heatmap(df.drop(columns="Sexo").corr(), annot=True, cmap="coolwarm", fmt=".2f", center=0)
plt.title("Matriz de correlacion"); plt.tight_layout(); plt.show()

## 3. Configuración del entorno PyCaret

Se normalizan los datos, se elimina multicolinealidad (umbral 0.9), se aplican transformaciones
(Yeo-Johnson) y se remueven outliers. Validación cruzada de 10 folds y 70% de entrenamiento.

In [ ]:
reg = setup(
    df,
    target=TARGET,
    session_id=123,
    normalize=True,
    remove_multicollinearity=True,
    multicollinearity_threshold=0.9,
    transformation=True,
    remove_outliers=True,
    train_size=0.7,
    fold=10,
)

## 4. Comparación y selección de modelos

In [ ]:
best_model = compare_models()

In [ ]:
# Random Forest como modelo base a optimizar
rf = create_model("rf")

In [ ]:
# Afinado priorizando primero MAPE y luego MAE
tuned_mape = tune_model(rf, optimize="MAPE")
tuned_mae = tune_model(rf, optimize="MAE")

## 5. Evaluación del modelo

In [ ]:
plot_model(tuned_mape, plot="residuals")
plot_model(tuned_mape, plot="error")
plot_model(tuned_mape, plot="feature")

In [ ]:
# Predicciones sobre el conjunto de prueba
predictions = predict_model(tuned_mape)

y_real = get_config("y_test")
y_pred = predictions["prediction_label"]

plt.figure(figsize=(7, 6))
plt.scatter(y_real, y_pred, alpha=0.7)
lim = [min(y_real.min(), y_pred.min()), max(y_real.max(), y_pred.max())]
plt.plot(lim, lim, "--", color="gray")
plt.xlabel("Valores reales"); plt.ylabel("Predicciones")
plt.title("Real vs. Predicho (% total primera)"); plt.tight_layout(); plt.show()

In [ ]:
evaluate_model(tuned_mape)

## 6. Conclusiones

- PyCaret compara múltiples regresores; **Random Forest** ofrece el mejor equilibrio y se afina por
  MAPE/MAE (métricas de error relativo apropiadas para producción).
- Las gráficas de residuos, error e **importancia de variables** permiten interpretar qué factores de
  crecida (pesos por día, ICA/IEP) más influyen en el rendimiento de la campaña.

**Próximos pasos:** validación con datos de nuevas campañas, comparar contra Gradient Boosting/XGBoost
y registrar el modelo para predicción en producción.